# EDA sobre discapacidad reconocida en España

Este proyecto analiza la evolución y distribución de los registros de discapacidad reconocida en España entre 2019 y 2024.

El objetivo principal es explorar los datos, limpiarlos si es necesario, visualizar patrones relevantes y extraer conclusiones comprensibles.

In [2]:
from pathlib import Path
import requests

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, ColumnDataSource

output_notebook()

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

sns.set_theme(style="whitegrid")

Loading BokehJS ...

## Carga del dataset

El archivo `bdepcd_2024.csv` se encuentra incluido en este repositorio.  
Contiene datos públicos sobre discapacidad reconocida en España entre 2019 y 2024.

In [4]:
DATA_URL = "https://raw.githubusercontent.com/Bootcamp-IA-MAD-P7/project3-EDA-Fernanda/main/bdepcd_2024.csv"
DATA_PATH = Path("bdepcd_2024.csv")

if not DATA_PATH.exists():
    response = requests.get(DATA_URL, timeout=30)
    response.raise_for_status()
    DATA_PATH.write_bytes(response.content)

dtypes = {
    "Año": "int16",
    "Comunidad_Autónoma": "category",
    "Provincia": "category",
    "Grupo_DIAG0": "category",
    "Sexo": "category",
    "Grupo_grado_discapacidad": "category",
    "Grupo_edad": "category",
    "Valor": "int32",
}

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig", sep=";", dtype=dtypes)
df.head()

,Año,Comunidad_Autónoma,Provincia,Grupo_DIAG0,Sexo,Grupo_grado_discapacidad,Grupo_edad,Valor
0,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 18 a 34 años,33
1,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 35 a 64 años,367
2,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 65 a 79 años,397
3,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 7 a 17 años,6
4,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 80 años o más,164


In [5]:
df.shape

(148626, 8)

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 148626 entries, 0 to 148625
Data columns (total 8 columns):
 #   Column                    Non-Null Count   Dtype   
---  ------                    --------------   -----   
 0   Año                       148626 non-null  int16   
 1   Comunidad_Autónoma        148626 non-null  category
 2   Provincia                 148626 non-null  category
 3   Grupo_DIAG0               148626 non-null  category
 4   Sexo                      148626 non-null  category
 5   Grupo_grado_discapacidad  148626 non-null  category
 6   Grupo_edad                148626 non-null  category
 7   Valor                     148626 non-null  int32   
dtypes: category(6), int16(1), int32(1)
memory usage: 1.7 MB


In [ ]:
df.head()

,Año,Comunidad_Autónoma,Provincia,Grupo_DIAG0,Sexo,Grupo_grado_discapacidad,Grupo_edad,Valor
0,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 18 a 34 años,33
1,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 35 a 64 años,367
2,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 65 a 79 años,397
3,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 7 a 17 años,6
4,2019,Andalucía,Almería,"Cardiovascular, hematológica, inmunológica o r...",H,De 33% a 44%,De 80 años o más,164


In [7]:
df.isnull().sum()

Año                         0
Comunidad_Autónoma          0
Provincia                   0
Grupo_DIAG0                 0
Sexo                        0
Grupo_grado_discapacidad    0
Grupo_edad                  0
Valor                       0
dtype: int64

In [8]:
df.describe(include="all")

,Año,Comunidad_Autónoma,Provincia,Grupo_DIAG0,Sexo,Grupo_grado_discapacidad,Grupo_edad,Valor
count,148626.000000,148626,148626,148626,148626,148626,148626,148626.000000
unique,NaN,19,52,10,2,5,6,NaN
top,NaN,Andalucía,Madrid,Otra deficiencia,H,De 33% a 44%,De 35 a 64 años,NaN
freq,NaN,24355,3294,18350,75050,31833,27908,NaN
mean,2021.537234,NaN,NaN,NaN,NaN,NaN,NaN,187.013006
std,1.716856,NaN,NaN,NaN,NaN,NaN,NaN,595.940228
min,2019.000000,NaN,NaN,NaN,NaN,NaN,NaN,1.000000
25%,2020.000000,NaN,NaN,NaN,NaN,NaN,NaN,6.000000
50%,2022.000000,NaN,NaN,NaN,NaN,NaN,NaN,33.000000
75%,2023.000000,NaN,NaN,NaN,NaN,NaN,NaN,138.000000


In [9]:
df.duplicated().sum()

np.int64(0)

## Comprensión inicial del dataset

Antes de limpiar o analizar los datos, es importante entender qué representa cada fila.

Cada fila del dataset representa una combinación específica de características: año, comunidad autónoma, provincia, grupo diagnóstico, sexo, grado de discapacidad y grupo de edad.

La columna `Valor` indica la cantidad de personas que cumplen con esa combinación de características.

Por ejemplo, una fila puede representar cuántas personas de una provincia determinada, de un sexo concreto, dentro de un grupo de edad específico y con un determinado grado de discapacidad, están registradas en un año concreto.

Por este motivo, para analizar cantidades totales de personas no se deben contar filas, sino sumar la columna `Valor`.

In [ ]:
df_limpio = df.copy()

## Limpieza y preprocesamiento de datos

Antes de comenzar el análisis, se revisa si el dataset contiene valores nulos, registros duplicados o tipos de datos que convenga ajustar.

En este caso, la base de datos ya se encuentra bastante estructurada, por lo que el preprocesamiento se enfocará en validar la calidad de los datos y optimizar algunos tipos de variables para facilitar el análisis posterior.

Se revisa si existen valores faltantes en alguna columna. Esto es importante porque cada columna aporta una condición necesaria para interpretar correctamente la cantidad registrada en `Valor`.

In [10]:
df_limpio.isnull().sum()

Año                         0
Comunidad_Autónoma          0
Provincia                   0
Grupo_DIAG0                 0
Sexo                        0
Grupo_grado_discapacidad    0
Grupo_edad                  0
Valor                       0
dtype: int64

No se observan valores nulos en el dataset. Esto indica que todas las filas cuentan con la información necesaria para ser interpretadas. Por lo que no es necesario eliminar ni imputar datos faltantes.

### Revisión de duplicados

Se revisa si existen filas duplicadas. Como cada fila representa una combinación específica de características, una fila repetida podría generar un doble conteo en el análisis.

In [11]:
df_limpio.duplicated().sum()

np.int64(0)

No se detectan registros duplicados. Por este motivo, se conserva la totalidad de las filas del dataset.

In [ ]:
## Preprocesamiento de datos

Después de comprobar que no existen valores nulos ni filas duplicadas, se realizan algunos ajustes para preparar el dataset para el análisis.

Estos ajustes no modifican el contenido de los datos, sino que permiten trabajar con tipos de variables más adecuados y mantener un orden lógico en variables como grupo de edad y grado de discapacidad.

In [12]:
df_limpio = df.copy()

df_limpio["Año"] = df_limpio["Año"].astype("int16")
df_limpio["Valor"] = df_limpio["Valor"].astype("int32")

La columna `Año` se convierte a tipo entero `int16`, ya que contiene valores pequeños correspondientes a años. La columna `Valor` se convierte a `int32`, porque representa cantidades de personas y puede contener números más altos.

In [14]:
columnas_categoricas = [
    "Comunidad_Autónoma",
    "Provincia",
    "Grupo_DIAG0",
    "Sexo",
    "Grupo_grado_discapacidad",
    "Grupo_edad"
]

for columna in columnas_categoricas:
    df_limpio[columna] = df_limpio[columna].astype("category")

Convierte Grupo_edad en una categoría ordenada, usando este orden lógico.

In [15]:
orden_edad = [
    "Menos de 7 años",
    "De 7 a 17 años",
    "De 18 a 34 años",
    "De 35 a 64 años",
    "De 65 a 79 años",
    "De 80 años o más"
]

df_limpio["Grupo_edad"] = pd.Categorical(
    df_limpio["Grupo_edad"],
    categories=orden_edad,
    ordered=True
)

In [ ]:
Convierte Grupo_grado_discapacidad en una categoría ordenada, usando este orden lógico.

In [16]:
orden_grado = [
    "Menos de 33%",
    "De 33% a 44%",
    "De 45% a 63%",
    "De 64% a 74%",
    "De 75% a 100%"
]

df_limpio["Grupo_grado_discapacidad"] = pd.Categorical(
    df_limpio["Grupo_grado_discapacidad"],
    categories=orden_grado,
    ordered=True
)

In [17]:
df_limpio.dtypes

Año                            int16
Comunidad_Autónoma          category
Provincia                   category
Grupo_DIAG0                 category
Sexo                        category
Grupo_grado_discapacidad    category
Grupo_edad                  category
Valor                          int32
dtype: object

Se ajustan los tipos de datos para preparar el dataset antes del análisis. Las columnas numéricas se convierten a tipos enteros más eficientes, mientras que las columnas descriptivas se transforman en variables categóricas. Además, se define un orden lógico para los grupos de edad y los grados de discapacidad, ya que estas variables tienen una secuencia natural que no coincide necesariamente con el orden alfabético.

### Renombrado de columnas

Algunas columnas tienen nombres poco intuitivos para el análisis. Por este motivo, se renombran para que su significado sea más claro durante el desarrollo del EDA.

En particular, `Grupo_DIAG0` se renombra como `Tipo_discapacidad`, ya que contiene categorías amplias sobre el tipo de discapacidad o deficiencia reconocida. Además, `Valor` se renombra como `Cantidad_personas`, porque representa el número de personas dentro de cada combinación de características.

Renombrar columnas no cambia los datos. Solo mejora la legibilidad del análisis.

In [18]:
df_limpio = df_limpio.rename(columns={
    "Grupo_DIAG0": "Tipo_discapacidad",
    "Valor": "Cantidad_personas"
})

In [19]:
df_limpio.columns

Index(['Año', 'Comunidad_Autónoma', 'Provincia', 'Tipo_discapacidad', 'Sexo',
       'Grupo_grado_discapacidad', 'Grupo_edad', 'Cantidad_personas'],
      dtype='str')